# Train YOLO26 — Hi-Vis Vest Compliance (2 classes)

Fine-tunes **`yolo26m.pt`** on the `ICP_hi-vis-vest_yolo_dataset` (classes: `person_no_vest`, `person_vest`).

**Before you run:** Runtime → Change runtime type → **GPU** (T4 is fine).

Run the cells top to bottom. The only thing you may need to edit is `DRIVE_DATASET` in the Config cell if auto-detect fails.

## 1. Check GPU

In [ ]:
!nvidia-smi
# If this errors or shows no GPU: Runtime > Change runtime type > Hardware accelerator = GPU

## 2. Install Ultralytics (includes YOLO26)

In [ ]:
%pip install -U ultralytics

import ultralytics, torch
ultralytics.checks()
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Config + locate the dataset on Drive

This auto-searches your Drive for either `ICP_hi-vis-vest_yolo_dataset.zip` or the extracted folder.
If it can't find it, set `DRIVE_DATASET` manually to the exact path (right-click the file/folder in the Colab file browser → Copy path).

In [ ]:
import os, glob, zipfile, shutil, time

# ---- Optional manual override: paste the exact path to the .zip OR the folder ----
DRIVE_DATASET = None  # e.g. '/content/drive/MyDrive/ICP_hi-vis-vest_yolo_dataset.zip'

LOCAL_DATASET = '/content/dataset'  # fast local disk for training (do NOT train off Drive)
NAME = 'ICP_hi-vis-vest_yolo_dataset'

def find_dataset():
    if DRIVE_DATASET:
        return DRIVE_DATASET
    roots = ['/content/drive/MyDrive', '/content/drive/Shareddrives']
    # Prefer an already-extracted folder (has images/train), else the zip
    for r in roots:
        for p in glob.glob(f'{r}/**/{NAME}', recursive=True):
            if os.path.isdir(p) and os.path.isdir(os.path.join(p, 'images', 'train')):
                return p
    for r in roots:
        for p in glob.glob(f'{r}/**/{NAME}.zip', recursive=True):
            return p
    return None

src = find_dataset()
assert src, (f'Could not find {NAME} on Drive. Set DRIVE_DATASET manually above.')
print('Found dataset source:', src)

# Prepare a clean local copy
if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)
os.makedirs(LOCAL_DATASET, exist_ok=True)

t0 = time.time()
if src.endswith('.zip'):
    print('Unzipping to local disk (this takes a few minutes for ~1.6 GB)...')
    with zipfile.ZipFile(src) as z:
        z.extractall(LOCAL_DATASET)
else:
    print('Copying folder to local disk...')
    shutil.copytree(src, os.path.join(LOCAL_DATASET, NAME))
print(f'Done in {time.time()-t0:.0f}s')

# Find the folder that actually contains images/train (handles nested zips)
root = None
for dirpath, dirnames, _ in os.walk(LOCAL_DATASET):
    if os.path.isdir(os.path.join(dirpath, 'images', 'train')):
        root = dirpath
        break
assert root, 'Extracted, but could not locate images/train. Check the archive layout.'
print('Dataset root:', root)

## 5. Write a correct `data.yaml`

The one in the dataset points at a hardcoded local Mac path, so we regenerate it pointing at the local Colab copy.

In [ ]:
import yaml

data = {
    'path': root,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 2,
    'names': {0: 'person_no_vest', 1: 'person_vest'},
}
DATA_YAML = '/content/data.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(open(DATA_YAML).read())

## 6. Sanity check — image/label counts must match

In [ ]:
from collections import Counter

def count(split):
    imgs = [f for f in glob.glob(f'{root}/images/{split}/*') if not f.endswith('.DS_Store')]
    lbls = [f for f in glob.glob(f'{root}/labels/{split}/*.txt')]
    cls = Counter()
    empty = 0
    for l in lbls:
        lines = [ln for ln in open(l).read().splitlines() if ln.strip()]
        if not lines:
            empty += 1
        for ln in lines:
            cls[int(ln.split()[0])] += 1
    print(f'{split}: {len(imgs)} images, {len(lbls)} labels, {empty} background(empty) | boxes {dict(cls)}')

count('train')
count('val')

## 7. Train

Starts from COCO-pretrained `yolo26m.pt` (already knows "person"). ~100 epochs; early-stops if val stops improving.
Drop `batch` to 8 if you hit CUDA out-of-memory on a small GPU.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26m.pt')  # auto-downloads pretrained weights

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,          # lower to 8 if out-of-memory
    patience=20,       # early stopping
    device=0,
    cache='disk',      # faster epochs; images decoded once
    project='/content/runs',
    name='hivis_yolo26m',
    plots=True,
    seed=0,
)
print('Best weights:', model.trainer.best)

## 8. Validate + view training curves

In [ ]:
from IPython.display import Image, display

best = model.trainer.best  # path to best.pt
best_model = YOLO(best)
metrics = best_model.val(data=DATA_YAML, split='val')
print(f"mAP50-95: {metrics.box.map:.3f} | mAP50: {metrics.box.map50:.3f}")

run_dir = os.path.dirname(best).replace('/weights', '')
for img in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = os.path.join(run_dir, img)
    if os.path.exists(p):
        print(img); display(Image(filename=p, width=760))

## 9. Try it on a few validation images

In [ ]:
import random

val_imgs = [f for f in glob.glob(f'{root}/images/val/*') if not f.endswith('.DS_Store')]
sample = random.sample(val_imgs, min(6, len(val_imgs)))
preds = best_model.predict(sample, imgsz=640, conf=0.25, save=True, project='/content/runs', name='predict')
for r in preds:
    display(Image(filename=r.save_dir + '/' + os.path.basename(r.path), width=640))

## 10. Save results back to Drive

In [ ]:
OUT = '/content/drive/MyDrive/YOLO26_hivis_results'
os.makedirs(OUT, exist_ok=True)

# Copy the best weights (renamed) and the whole run folder (plots, metrics, args)
shutil.copy(best, os.path.join(OUT, 'best.pt'))
dst_run = os.path.join(OUT, os.path.basename(run_dir))
if os.path.exists(dst_run):
    shutil.rmtree(dst_run)
shutil.copytree(run_dir, dst_run)
print('Saved to Drive:', OUT)
print(os.listdir(OUT))